# NB_02 — Electroplated Bi Process Window v2

**Engineering question**

> What electroplated-bismuth thickness and microstructure process window preserves Gaussian-like spectral response while increasing x-ray stopping power?

This notebook uses **Engineering Objects** as its primary interface and pulls quantitative evidence from the completed source records where needed.

It does **not** invent an optimized process window where the repository lacks measurements. Instead, it separates:

- source-supported operating points;
- source-supported detector outcomes;
- unresolved process variables;
- candidate process-window dimensions;
- the next measurements needed to establish quantitative tolerances.

Run from top to bottom.


## Workflow

```text
Engineering Objects
        ↓
Source-supported quantitative evidence
        ↓
Electroplating variables
        ↓
Coupled absorber + TES variables
        ↓
Observed operating points
        ↓
Candidate process-window dimensions
        ↓
Validation matrix
        ↓
Engineering outputs
```


## 1. Locate repository and load Engineering Objects

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]

NOTEBOOK_ID = "NB_02_ELECTROPLATED_BI_PROCESS_WINDOW"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / "engineering_navigator").is_dir()
        ):
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")

        if target.exists():
            if (target / "engineering_navigator").is_dir():
                return target
            raise FileExistsError(
                f"{target} exists but does not look like sensors-becker."
            )

        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(target)],
            check=True,
        )

        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker repository. "
        "Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OBJ_DIR = ROOT / "engineering_navigator" / "engineering_objects"
SOURCE_DIR = (
    ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
OUTPUT_DIR = (
    ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / "PROCESS_WINDOW_01"
)
EXPORT_DIR = ROOT / "exports" / "PROCESS_WINDOW_01"
EXPORT_ZIP = ROOT / "exports" / "PROCESS_WINDOW_01_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)


def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected one top-level YAML mapping")

    return data


def load_object(name: str) -> dict:
    return load_yaml(OBJ_DIR / f"{name}.yaml")


absorber = load_object("absorber")
electroplating = load_object("electroplating")
tes = load_object("tes")

print(f"Repository     : {ROOT}")
print(f"Object directory: {OBJ_DIR.relative_to(ROOT)}")
print("Loaded objects : absorber, electroplating, tes")


## 2. Load completed source records

In [ ]:
records = {}

for filename in SOURCE_FILES:
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(f"{filename}: missing source_id")

    records[source_id] = record

incomplete = {
    source_id: record.get("extraction_status")
    for source_id, record in records.items()
    if not str(record.get("extraction_status", "")).startswith("complete")
}

if incomplete:
    raise ValueError(
        "Incomplete source records: " + json.dumps(incomplete, indent=2)
    )

print("Source-record validation: PASS")
print("Loaded:", ", ".join(sorted(records)))


## 3. Electroplating process variables from the Engineering Object

In [ ]:
process_rows = []

for variable in electroplating.get("variables", []):
    process_rows.append({
        "variable": variable.get("id"),
        "unit": variable.get("unit", ""),
        "current_numeric_evidence": "",
        "candidate_range": "",
        "range_status": "not_yet_established",
    })

process_df = pd.DataFrame(process_rows)
process_df


## 4. Extract source-supported quantitative evidence

This table is constructed from `reported_values` in the completed source records. It keeps the source identity attached to each value so the notebook can distinguish measured operating points from future candidate ranges.


In [ ]:
value_rows = []

for source_id, record in records.items():
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue

        value_rows.append({
            "source_id": source_id,
            "object": item.get("object"),
            "variable": item.get("variable"),
            "value": item.get("value"),
            "unit": item.get("unit"),
            "condition": item.get("condition"),
            "source_page": item.get("source_page"),
        })

values_df = pd.DataFrame(value_rows)

PROCESS_VALUE_ALIASES = {
    "Bi_thickness",
    "current_density",
    "bias_voltage",
    "plating_rate",
    "Au_seed_thickness",
    "average_grain_size",
    "average_grain_radius",
    "diffraction_grain_size",
    "SEM_grain_size",
    "quantum_efficiency",
    "C",
    "G",
    "delta_E",
    "predicted_delta_E",
    "residual_resistance_ratio",
    "cloud_size",
}

quantitative_evidence = (
    values_df[values_df["variable"].isin(PROCESS_VALUE_ALIASES)]
    .sort_values(["variable", "source_id", "object"])
    .reset_index(drop=True)
)

quantitative_evidence


## 5. Source-supported electroplating operating point

In [ ]:
operating_point_rows = []

# Pull process parameters from the electroplating Engineering Object where available.
for point in electroplating.get("reported_process_points", []):
    row = {
        "source_id": point.get("source"),
        "current_density_mA_per_cm2": point.get("current_density_mA_per_cm2"),
        "bias_voltage_V": point.get("bias_voltage_V"),
        "plating_rate_nm_per_min": point.get("plating_rate_nm_per_min"),
        "bismuth_thickness_um": point.get("bismuth_thickness_um"),
        "bath_temperature": point.get("bath_temperature"),
        "agitation": point.get("agitation"),
    }
    operating_point_rows.append(row)

operating_points_df = pd.DataFrame(operating_point_rows)

if operating_points_df.empty:
    print("No electroplating operating point recorded in electroplating.yaml.")
else:
    operating_points_df


## 6. Coupled absorber and TES variables

In [ ]:
coupled_rows = []

for object_name, obj in [
    ("absorber", absorber),
    ("tes", tes),
]:
    for variable in obj.get("variables", []):
        coupled_rows.append({
            "engineering_object": object_name,
            "variable": variable.get("id"),
            "unit": variable.get("unit", ""),
        })

coupled_df = (
    pd.DataFrame(coupled_rows)
    .drop_duplicates()
    .sort_values(["engineering_object", "variable"])
    .reset_index(drop=True)
)

coupled_df


## 7. Current evidence constraints

These are **evidence-supported constraints**, not final manufacturing tolerances.


In [ ]:
constraint_rows = []

for source_id, record in records.items():
    for item in record.get("engineering_constraints", []):
        if not isinstance(item, dict):
            continue

        constraint_rows.append({
            "source_id": source_id,
            "constraint": item.get("constraint"),
            "specification": item.get("specification"),
            "source_pages": item.get("source_pages"),
        })

constraints_df = pd.DataFrame(constraint_rows)

FOCUS_CONSTRAINT_KEYWORDS = (
    "spectral",
    "quantum",
    "heat capacity",
    "thermal",
    "energy resolution",
    "microstructure",
    "absorber",
)

if not constraints_df.empty:
    mask = constraints_df.apply(
        lambda row: any(
            keyword in (
                f"{row.get('constraint', '')} "
                f"{row.get('specification', '')}"
            ).lower()
            for keyword in FOCUS_CONSTRAINT_KEYWORDS
        ),
        axis=1,
    )
    process_constraints_df = constraints_df[mask].reset_index(drop=True)
else:
    process_constraints_df = constraints_df

process_constraints_df


## 8. Candidate process-window dimensions

In [ ]:
window_dimensions = pd.DataFrame([
    {
        "dimension": "Bi_thickness",
        "engineering_role": "stopping power + thermalization",
        "current_evidence": "3 µm appears in SOURCE_01 and SOURCE_02",
        "candidate_range": "",
        "range_status": "requires_controlled_thickness_series",
    },
    {
        "dimension": "grain_size",
        "engineering_role": "carrier thermalization + spectral tail",
        "current_evidence": "electroplated Bi has much larger grains than evaporated Bi",
        "candidate_range": "",
        "range_status": "acceptance_threshold_not_established",
    },
    {
        "dimension": "current_density",
        "engineering_role": "electroplating process control",
        "current_evidence": "one reported operating point",
        "candidate_range": "",
        "range_status": "process_tolerance_not_established",
    },
    {
        "dimension": "bias_voltage",
        "engineering_role": "electroplating process control",
        "current_evidence": "one reported operating point",
        "candidate_range": "",
        "range_status": "process_tolerance_not_established",
    },
    {
        "dimension": "plating_rate",
        "engineering_role": "deposition kinetics + microstructure",
        "current_evidence": "one reported operating point",
        "candidate_range": "",
        "range_status": "process_tolerance_not_established",
    },
])

window_dimensions


## 9. Validation experiment matrix

The current source set supports designing the measurements needed to establish a process window, but not yet a numeric tolerance range.


In [ ]:
validation_matrix = pd.DataFrame([
    {
        "input_variable": "Bi_thickness",
        "strategy": "controlled sweep",
        "held_constant": "electroplating chemistry + TES + membrane",
        "primary_response": "low_energy_tail_fraction",
        "secondary_responses": "quantum_efficiency, C, energy_resolution",
        "measurement": "x-ray spectrum + thermal characterization",
    },
    {
        "input_variable": "current_density",
        "strategy": "controlled sweep around reported operating point",
        "held_constant": "Bi_thickness + bath + geometry",
        "primary_response": "grain_size_distribution",
        "secondary_responses": "morphology, spectral response",
        "measurement": "SEM / diffraction + TES spectrum",
    },
    {
        "input_variable": "bias_voltage",
        "strategy": "controlled sweep around reported operating point",
        "held_constant": "Bi_thickness + bath + geometry",
        "primary_response": "grain_size_distribution",
        "secondary_responses": "morphology, spectral response",
        "measurement": "SEM / diffraction + TES spectrum",
    },
    {
        "input_variable": "plating_rate",
        "strategy": "controlled sweep",
        "held_constant": "Bi_thickness + bath + geometry",
        "primary_response": "grain_size_distribution",
        "secondary_responses": "roughness, spectral response",
        "measurement": "SEM / diffraction + TES spectrum",
    },
    {
        "input_variable": "batch",
        "strategy": "replicated production batches",
        "held_constant": "nominal process recipe",
        "primary_response": "repeatability",
        "secondary_responses": "yield, grain size, tail fraction",
        "measurement": "multi-wafer statistical characterization",
    },
])

validation_matrix


## 10. Process-window status

In [ ]:
process_window_status = {
    "notebook_id": NOTEBOOK_ID,
    "status": "measurement_plan_ready",
    "numeric_process_window_established": False,
    "source_supported_operating_points": len(operating_points_df),
    "leading_process_variables": [
        "Bi_thickness",
        "grain_size",
        "current_density",
        "bias_voltage",
        "plating_rate",
    ],
    "leading_detector_responses": [
        "low_energy_tail_fraction",
        "quantum_efficiency",
        "energy_resolution",
        "C",
        "G",
    ],
    "next_engineering_step": (
        "Acquire or identify replicated measurements that connect "
        "electroplating parameters to grain-size distributions and detector response."
    ),
}

process_window_status


## 11. Write outputs and export

In [ ]:
process_variables_csv = OUTPUT_DIR / "process_variables.csv"
quantitative_evidence_csv = OUTPUT_DIR / "quantitative_evidence.csv"
operating_points_csv = OUTPUT_DIR / "operating_points.csv"
coupled_variables_csv = OUTPUT_DIR / "coupled_variables.csv"
constraints_csv = OUTPUT_DIR / "process_constraints.csv"
window_dimensions_csv = OUTPUT_DIR / "candidate_process_window.csv"
validation_matrix_csv = OUTPUT_DIR / "validation_matrix.csv"
status_json = OUTPUT_DIR / "process_window_status.json"

process_df.to_csv(process_variables_csv, index=False)
quantitative_evidence.to_csv(quantitative_evidence_csv, index=False)
operating_points_df.to_csv(operating_points_csv, index=False)
coupled_df.to_csv(coupled_variables_csv, index=False)
process_constraints_df.to_csv(constraints_csv, index=False)
window_dimensions.to_csv(window_dimensions_csv, index=False)
validation_matrix.to_csv(validation_matrix_csv, index=False)
status_json.write_text(
    json.dumps(process_window_status, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

written_files = {
    "process_variables": process_variables_csv,
    "quantitative_evidence": quantitative_evidence_csv,
    "operating_points": operating_points_csv,
    "coupled_variables": coupled_variables_csv,
    "process_constraints": constraints_csv,
    "candidate_process_window": window_dimensions_csv,
    "validation_matrix": validation_matrix_csv,
    "process_window_status": status_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(ROOT)}")


In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 12. Engineering handoff

Version 2 establishes the first **object-driven quantitative engineering notebook**.

The current repository supports:

- a source-supported electroplating operating point;
- a source-supported microstructure / detector-response relationship;
- coupled absorber and TES variables;
- a concrete validation matrix.

It does **not yet support a numeric manufacturing tolerance window** because replicated process-to-response measurements are still missing.

The next substantive work is therefore to add evidence that varies process parameters across multiple devices or batches, then rerun this notebook to convert those measurements into candidate numerical ranges.

*Admissible generalizations trail leading specifications.*
